In [11]:
import anthropic
import dotenv
print("Both installed!")

# Load env variables
from dotenv import load_dotenv

load_dotenv()
print("Env variables loaded!")

# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-6"
print("Client created!")

Both installed!
Env variables loaded!
Client created!


In [12]:
#Helper Functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat (messages, system=None, temperature=0.7, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text
print("Helper functions defined!")

Helper functions defined!


In [13]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation.
The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for Python.
Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.
The tasks should be focused on Python use cases that would be useful to create a trading bot.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "python/json/regex",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    text = chat(messages, system="Respond with raw JSON only. No markdown, no explanation.")
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1]
        text = text.rsplit("```", 1)[0].strip()
    return json.loads(text)

In [21]:
dataset = generate_dataset()

with (open("005_dataset.json", "w")) as f:
    json.dump(dataset, f, indent=2)
print(json.dumps(dataset, indent=2))


[
  {
    "task": "Write a Python function that takes a list of closing prices and returns the Simple Moving Average (SMA) for a given window size.",
    "format": "python"
  },
  {
    "task": "Write a JSON object representing a trading bot configuration that includes fields for exchange name, trading pair, order type, risk percentage per trade, and stop loss percentage.",
    "format": "json"
  },
  {
    "task": "Write a regular expression that matches a valid trading pair symbol such as BTC/USDT, ETH/BTC, or SOL/USD, where both the base and quote currencies are 2 to 5 uppercase letters separated by a forward slash.",
    "format": "regex"
  }
]


In [15]:
def run_prompt(test_case):
    """Merges the prompt and the test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case['task']}

* Respond only with Python, JSON or a plain Regex.
* Do not include any explanations or additional text.
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [16]:
import re

def grade_by_model(test_case, output):
    eval_prompt = f"""
Please grade the following output on a scale of 1 to 10, with 10 being the best.
The output should be graded based on how well it solves the task, and how well it follows the instructions.
Include a breif reasoning for your grading.

Original Task:
<task>
{test_case['task']}
</task>

Solution to evaluate:
<solution>
{output}
</solution>

Output Format:
Provide your grading in a structured JSON format with the following fields, in this exact order:
- "strengths": A brief description of the strengths of the solution.
- "weaknesses": A brief description of the weaknesses of the solution.
- "reasoning": A brief explanation of the reasoning behind the score.
- "score": A numeric score from 1 to 10, with 10 being the best.

Respond with only the JSON. Keep your response concise and to the point.
Use plain English only in all string values. Do not include code, regex patterns, or backslashes.
Example response:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}

"""

    messages = []
    add_user_message(messages, eval_prompt)
    eval_text = chat(messages, system="Respond with raw JSON only. No markdown, no explanation. Do not use backslashes in any string values.")
    text = eval_text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1]
        text = text.rsplit("```", 1)[0].strip()
    text = re.sub(r'\\(?!["\\/bfnrtu])', r'\\\\', text)
    return json.loads(text)

In [17]:
import re
import ast

def strip_fences(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1]
        text = text.rsplit("```", 1)[0].strip()
    return text

def validate_json(text):
    text = strip_fences(text)
    if not text:
        return 1
    score = 0
    try:
        parsed = json.loads(text)
        score += 6
        if isinstance(parsed, (dict, list)) and len(parsed) > 0:
            score += 2
        if len(text) > 50:
            score += 2
    except json.JSONDecodeError:
        if text.startswith("{") or text.startswith("["):
            score += 2
    return min(score, 10)

def validate_python(text):
    text = strip_fences(text)
    if not text:
        return 1
    score = 0
    try:
        tree = ast.parse(text)
        score += 6
        if any(isinstance(n, ast.FunctionDef) for n in ast.walk(tree)):
            score += 2
        if len(text.splitlines()) > 3:
            score += 2
    except SyntaxError:
        if "def " in text or "import " in text:
            score += 2
    return min(score, 10)

def validate_regex(text):
    text = strip_fences(text)
    if not text:
        return 1
    score = 0
    try:
        re.compile(text)
        score += 6
        if len(text) > 5:
            score += 2
        if any(c in text for c in r"()[]{}^$"):
            score += 2
    except re.error:
        score += 1
    return min(score, 10)

def grade_syntax(response, test_case):
    fmt = test_case['format']
    if fmt == 'regex':
        return validate_regex(response)
    elif fmt == 'json':
        return validate_json(response)
    elif fmt == 'python':
        return validate_python(response)
    else:
        raise ValueError(f"Unknown format: {fmt}")


In [18]:
def run_test_case(test_case):
    """Calls run_prompt with the test case, then evaluates the result and returns a score"""
    output = run_prompt(test_case)

    model_result = grade_by_model(test_case, output)
    model_score = model_result["score"]
    reasoning = model_result["reasoning"]
    strengths = model_result["strengths"]
    weaknesses = model_result["weaknesses"]

    code_score = grade_syntax(output, test_case)

    score = (model_score + code_score) / 2


    return {
        "output": output,
        "test_case": test_case,
        "model score": model_score,
        "code score": code_score,
        "score": score,
        "reasoning": reasoning,
        "strengths": strengths,
        "weaknesses": weaknesses
    }

In [19]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case for each test case"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean(result["score"] for result in results)
    print(f"Average Score: {average_score}")

    return results


In [20]:
with  open("005_dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
print(json.dumps(results, indent=2))

Average Score: 9.333333333333334
[
  {
    "output": "```python\ndef calculate_sma(prices: list, window: int) -> list:\n    if not prices or window <= 0 or window > len(prices):\n        return []\n    \n    sma_values = []\n    for i in range(len(prices) - window + 1):\n        window_slice = prices[i:i + window]\n        sma = sum(window_slice) / window\n        sma_values.append(round(sma, 2))\n    \n    return sma_values\n```",
    "test_case": {
      "task": "Write a Python function that takes a list of closing prices and returns the Simple Moving Average (SMA) for a given window size.",
      "format": "python"
    },
    "model score": 8,
    "code score": 10,
    "score": 9.0,
    "reasoning": "The solution correctly solves the core task of computing SMA for a given window size with proper edge case handling. The implementation is clean and readable. Minor issues include the opinionated rounding behavior and lack of documentation, but overall it is a solid and functional solut